# MP22 — Upsampling Architecture Comparison

Upsampling can interpolate features, spread features with a transposed convolution, or rearrange channels with PixelShuffle. All three create a larger spatial tensor, but they differ in parameters, computation and phase behavior.

## Mathematics, shapes and worked example

PixelShuffle rearranges N×(C r²)×H×W into N×C×rH×rW. For C=3,r=2, twelve channels become three. A transposed-convolution output length is (n−1)s−2p+d(k−1)+output_padding+1. Here n=8,s=2,p=1,d=1,k=3,output_padding=1 gives 16. This is not a matrix inverse of convolution. Kernel/stride overlap can differ across output phases, creating checkerboard patterns.

In [ ]:
from pathlib import Path
import sys,json,torch
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(ROOT/'src'))
torch.manual_seed(19)
from mp22_tools import UpsamplingCNN
x=torch.rand(1,3,5,7)
for mode in ['shuffle','transpose','interpolate']:
 m=UpsamplingCNN(mode)
 assert m(x).shape==(1,3,10,14)
 print(mode,sum(p.numel() for p in m.parameters()))
z=torch.arange(4.).reshape(1,4,1,1)
assert torch.equal(torch.nn.PixelShuffle(2)(z),torch.tensor([[[[0.,1.],[2.,3.]]]]))

## Reproducible experiment

Run `python scripts/cnn_lessons_experiment.py` from the repository root. The script first checks a small synthetic input/output shape. It then uses four training and four validation NAIP scenes, split before tiling: 16 nonoverlapping 32×32 RGB patches per scene. HR stored DN are divided by 10000, not claimed as reflectance. SR inputs use Gaussian sigma 0.8 HR pixels and area x2 downsampling. Identity reconstruction uses HR as both input and target.

Training: CPU float32, two threads, seed 19, Adam learning rate 0.001, batch four, 300 updates, MSE, no augmentation or scheduler. Evaluate the final fixed step; no validation-based checkpoint search. Teaching weights are regenerated and not committed. The constant baseline uses training-channel means. SR baselines use bicubic with align_corners=False. Metrics use range one, border two, all RGB channels; equal patches per scene give equal scene weighting.

Data: existing checksummed NAIP/OpenSR crops, public domain with USDA/OpenSR attribution. No new imagery download. Final research weights/protocol/test results are untouched by these separate teaching exercises. No learner mastery claim.

## Observed result and interpretation

Validation PSNR: PixelShuffle 41.632, interpolation-plus-convolution 41.190, transposed convolution 39.300 dB; bicubic 42.964 dB. Constant-input responses expose phase patterns. These panels use separate color ranges and are qualitative. Equal steps are not equal parameter count or compute; measured counts and warmed CPU times are printed below.

In [ ]:
report=json.loads((ROOT/'reports/mp19-mp22-cnn.json').read_text())
selected=['residual_shuffle', 'residual_transpose', 'residual_interpolate']
for row in report['results']:
 if row['model'] in selected:
  print({k:v for k,v in row.items() if k!='per_tile'})
from IPython.display import Image,display
display(Image(filename=str(ROOT/'reports/mp19-mp22-cnn.png')))
display(Image(filename=str(ROOT/'reports/mp22-phase-response.png')))

## Practice — solve first

1. How many pre-shuffle channels are needed for RGB x4?
2. Why is transposed convolution not an inverse?
3. Can PixelShuffle also exhibit checkerboards?

## Worked solutions

1. 3×4²=48 channels.
2. A learned transpose-shaped linear operator does not undo information lost by arbitrary downsampling.
3. Yes. Independently learned subpixel phases can disagree; rearrangement alone does not guarantee smoothness.

## Limitations and next question

One seed and a short budget describe an engineering demonstration. Interpolation, padding, input distribution and unequal parameter counts affect results. Investigate these factors on validation before any new frozen research test. Generated answers do not assess learner mastery.

[Authoritative brief](README.md) · [Local/Colab setup](../../README.md)